# Image → Jupyter Widget Dashboard (Scalable Template)

This notebook demonstrates a **modular**, **extensible** dashboard builder using `ipywidgets`.

**Highlights**
- Rebuilds a dashboard from a declarative spec (grid-based).
- Blocks: `title`, `subtitle`, `text`, `table`, `chart` (matplotlib, one plot, no colors), `image`, `metric`, `spacer`.
- Multi-tab support for future growth.
- Lightweight `DataModel` for static/real-time updates (pub/sub style).
- Export to standalone HTML using `ipywidgets.embed`.

> You can start from an uploaded image (slide/screenshot). We **do not** do heavy computer vision; instead, we generate a clean, editable starter spec you can tweak to match the layout.

In [4]:
from IPython.display import display
import ipywidgets as widgets
from image_dashboard_widget import (
    DataModel, LayoutSpec, BlockSpec, DashboardBuilder,
    parse_image_to_spec, build_dashboard_from_spec, export_dashboard_html,
    example_multitab_spec
)
import pandas as pd
import numpy as np

print("✅ Imported dashboard builder.")

✅ Imported dashboard builder.


## 1) Start from an uploaded image (optional)
Upload a slide/screenshot. We probe its aspect ratio (if `Pillow` is available) to choose a responsive grid, then build a starter spec for you to edit.

In [2]:
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader)

starter_spec = None
def _on_upload(change):
    global starter_spec
    for name, file_info in uploader.value.items():
        starter_spec = parse_image_to_spec(file_info['content'])
        print(f"Parsed starter spec from: {name}")

uploader.observe(_on_upload, names='value')

FileUpload(value=(), accept='image/*', description='Upload')

## 2) Or build from a manual spec (edit freely)
Everything is driven by a `LayoutSpec` with a list of `BlockSpec`s.

In [3]:
manual_spec = LayoutSpec(
    title="Operations Dashboard",
    ncols=4,
    min_col_px=240,
    blocks=[
        BlockSpec(type='title',    content='Production Overview'),
        BlockSpec(type='subtitle', content='Week 34 / Line A'),
        BlockSpec(type='metric',   content={'label':'Uptime','value':'99.8','suffix':'%'}),
        BlockSpec(type='metric',   content={'label':'Units/hr','value':'128'}),
        BlockSpec(type='chart',    content={'title':'Output (last 10)','x':list(range(10)),'y':[3,4,5,4,6,7,7,8,6,9]}),
        BlockSpec(type='table',    content={'Shift':['A','B','C'],'Count':[120,110,130]}),
        BlockSpec(type='text',     content='<b>Notes:</b> Replace blocks to match your layout.'),
    ]
)
manual_widget = build_dashboard_from_spec(manual_spec)
display(manual_widget)

TraitError: The 'align_items' trait of a Layout instance expected any of ['flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'unset'] (case-insensitive) or None, not the str 'start'.

## 3) Multi-tab layout (future scalability)
Use `example_multitab_spec()` as a starting point and extend.

In [ ]:
tabs_spec = example_multitab_spec()
tabs_widget = build_dashboard_from_spec(tabs_spec)
display(tabs_widget)

## 4) Data model integration (live/static updates)
Attach a `DataModel` to keep your blocks in sync with changing data sources.

In [ ]:
data = DataModel({
    'kpi': {'uptime': 99.9},
    'table': pd.DataFrame({'Item':['A','B','C'],'Qty':[1,2,3]}),
})

# Example: a dashboard that *reads* from this data (you would extend blocks to pull from data.data_key)
spec_with_data = LayoutSpec(
    title="Data-Driven",
    blocks=[
        BlockSpec(type='metric', content={'label':'Uptime','value':str(data['kpi']['uptime']),'suffix':'%'}, data_key='kpi'),
        BlockSpec(type='table',  content=data['table'], data_key='table'),
    ]
)
w_data = build_dashboard_from_spec(spec_with_data, data=data)
display(w_data)

def on_change(dm):
    # In a real app, you'd re-render or selectively update blocks that rely on dm keys
    print("Data changed:", dm._store.keys())

data.subscribe(on_change)
data.update({'kpi': {'uptime': 99.7}})  # triggers subscriber

## 5) Export to standalone HTML
This creates an embeddable HTML file you can share without a running kernel.

In [ ]:
export_path = export_dashboard_html(manual_widget, 'dashboard_export.html')
print('Exported to:', export_path)